In [126]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split


In [127]:
data = pd.read_csv(
    "Dataset/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)
data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [128]:
print(data.columns.tolist())

['label', 'message']


In [129]:
if "label" in data.columns and "message" in data.columns:
    data = data[["label", "message"]]
elif "v1" in data.columns and "v2" in data.columns:
    data = data[["v1", "v2"]]
    data.columns = ["label", "message"]
else:
    raise ValueError("Dataset must contain either " "['label', 'message'] or ['v1', 'v2'] columns.")
data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [130]:
print(data.isnull().sum())
data = data.dropna()

label      0
message    0
dtype: int64


In [131]:
print("Duplicates:", data.duplicated().sum())
data = data.drop_duplicates()

Duplicates: 403


In [132]:
data["label"] = data["label"].map({"ham": 0, "spam": 1})
data.head()
print(data["label"].value_counts())

label
0    4516
1     653
Name: count, dtype: int64


In [133]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

data["clean_message"] = data["message"].apply(clean_text)
data[["message", "clean_message"]].head()

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


In [134]:
X = data["clean_message"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4135
Testing samples: 1034


In [135]:
MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

print("Vocab size:", len(tokenizer.word_index))

Vocab size: 8328


In [136]:
X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

print(X_train_sequences[:2])

[[3354, 2, 64, 79, 149, 22, 4, 82, 42], [293, 273, 12, 277, 2224, 24, 7, 280, 142]]


In [137]:
X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training shape:", X_train_padded.shape)
print("Testing shape:", X_test_padded.shape)

Training shape: (4135, 100)
Testing shape: (1034, 100)


In [138]:
y_train = np.array(y_train)
y_test = np.array(y_test)

print(y_train.shape)
print(y_test.shape)


(4135,)
(1034,)


In [139]:
model = Sequential(
    [
        Embedding(input_dim=MAX_WORDS, output_dim=128),
        SimpleRNN(64, return_sequences=False),
        Dropout(0.4),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ]
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [140]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [141]:
log_dir = "logs/fit"

os.makedirs(log_dir, exist_ok=True)

early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [142]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train
)
class_weights = dict(zip(classes, class_weights_array))

In [143]:
history = model.fit(
    X_train_padded,
    y_train,
    validation_split=0.20,
    epochs=15,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping, tensorboard_callback],
    verbose=1,
)

Epoch 1/15


104/104 ━━━━━━━━━━━━━━━━━━━━ 18s 108ms/step - accuracy: 0.5973 - loss: 0.7038 - precision: 0.1698 - recall: 0.5545 - val_accuracy: 0.8755 - val_loss: 0.6382 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - accuracy: 0.7594 - loss: 0.5291 - precision: 0.3100 - recall: 0.7227 - val_accuracy: 0.9674 - val_loss: 0.1950 - val_precision: 0.8614 - val_recall: 0.8700
Epoch 3/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 10s 94ms/step - accuracy: 0.9157 - loss: 0.2972 - precision: 0.6182 - recall: 0.8863 - val_accuracy: 0.8585 - val_loss: 0.5253 - val_precision: 0.4581 - val_recall: 0.9300
Epoch 4/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 81ms/step - accuracy: 0.9764 - loss: 0.1813 - precision: 0.8891 - recall: 0.9313 - val_accuracy: 0.9565 - val_loss: 0.1957 - val_precision: 0.7623 - val_recall: 0.9300
Epoch 5/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.9752 - loss: 0.1457 - precision: 0.8696 - recall: 0.9479 - val_accuracy: 0.9553 - val_los

In [144]:
results = model.evaluate(X_test_padded, y_test, verbose=1)
print("Evaluation results:")
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9642 - loss: 0.1372 - precision: 0.8310 - recall: 0.9008
Evaluation results:
loss: 0.1372
compile_metrics: 0.9642


In [145]:
y_probability = model.predict(X_test_padded).ravel()
y_pred = (y_probability >= 0.5).astype(int)
print(y_pred[:20])

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step
[0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]


In [146]:
model.save("model.keras")

In [147]:
with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

In [148]:
def predict_message(message, threshold=0.5):
    cleaned = clean_text(message)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding="post", truncating="post")
    probability = model.predict(padded, verbose=0)[0][0]
    prediction = "SPAM" if probability >= threshold else "HAM"
    return prediction, probability

In [151]:
message = "Congratulations! You have won a free prize. Call now!"
prediction, probability = predict_message(message)

print("Message:", message)
print("Prediction:", prediction)
print(f"Spam Probability: {probability:.2%}")

Message: Congratulations! You have won a free prize. Call now!
Prediction: SPAM
Spam Probability: 96.42%


In [152]:
test_messages = [
    "Hey, are we meeting today?",
    "Congratulations! You won a free lottery prize. Call now!",
    "Can you send me the notes?",
    "URGENT! You have won £1000. Claim your prize now!",
    "I will be home at 8 pm.",
    "Free entry in a weekly competition! Text WIN to 80086.",
]

for message in test_messages:
    prediction, probability = predict_message(message)
    print("-" * 20)
    print("Message:", message)
    print("Prediction:", prediction)
    print(f"Spam Probability: {probability:.2%}")

--------------------
Message: Hey, are we meeting today?
Prediction: HAM
Spam Probability: 3.52%
--------------------
Message: Congratulations! You won a free lottery prize. Call now!
Prediction: SPAM
Spam Probability: 95.54%
--------------------
Message: Can you send me the notes?
Prediction: HAM
Spam Probability: 2.18%
--------------------
Message: URGENT! You have won £1000. Claim your prize now!
Prediction: SPAM
Spam Probability: 97.72%
--------------------
Message: I will be home at 8 pm.
Prediction: HAM
Spam Probability: 2.79%
--------------------
Message: Free entry in a weekly competition! Text WIN to 80086.
Prediction: SPAM
Spam Probability: 90.68%


In [153]:
test_messages = [
    "Congratulations! You have won a free prize. Call now!",
    "URGENT! You have won $1000. Claim your prize now!",
    "FREE entry to win a cash prize! Text WIN now!",
    "You have been selected to receive a free gift. Claim today!",
    "WINNER! You have won a £1000 cash prize. Call immediately!",
]

for message in test_messages:

    prediction, probability = predict_message(message)

    print("=" * 70)
    print("Message:", message)
    print("Prediction:", prediction)
    print(f"Spam Probability: {probability:.2%}")

Message: Congratulations! You have won a free prize. Call now!
Prediction: SPAM
Spam Probability: 96.42%
Message: URGENT! You have won $1000. Claim your prize now!
Prediction: SPAM
Spam Probability: 97.72%
Message: FREE entry to win a cash prize! Text WIN now!
Prediction: SPAM
Spam Probability: 96.55%
Message: You have been selected to receive a free gift. Claim today!
Prediction: SPAM
Spam Probability: 98.11%
Message: WINNER! You have won a £1000 cash prize. Call immediately!
Prediction: SPAM
Spam Probability: 97.92%
